# Chapter 11 — Few-Shot Optimization

**Book alignment:** DSPy From First Principles, Chapter 11

**Question this notebook isolates:** Does BootstrapFewShot's metric filter admit training traces while leaving a development-side regression outside its field of view?

In [ ]:
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported and constructed only; compile is never executed against a model

from common.data import canonical_split
from common.dspy_program import dspy_editorial_metric_v1
from common.metrics import editorial_metric

## The filter admits what the metric approves

BootstrapFewShot keeps a training trace only if the compile metric clears the threshold. Replay that admission rule deterministically over the chapter's inspected five-case prefix, using reference rewrites as stand-in traces — and confirm development evidence is never inspected.

In [ ]:
split = canonical_split()
by_id = {c.case_id: c for c in split.train}
optimizer = dspy.BootstrapFewShot(
    metric=dspy_editorial_metric_v1,
    metric_threshold=0.7,
    max_bootstrapped_demos=4,
    max_labeled_demos=0,
    max_rounds=1,
)

THRESHOLD = 0.7
inspected_ids = ["ed-001", "ed-002", "ed-005", "ed-006", "ed-007"]
admitted, rejected = [], []
for case_id in inspected_ids:
    case = by_id[case_id]
    score = editorial_metric(case, case.reference_rewrite, version="v1").score
    (admitted if score >= THRESHOLD else rejected).append((case_id, round(score, 3)))

unreached = [c for c in split.train_ids if c not in inspected_ids]
({"admitted": admitted, "rejected": rejected, "unreached": len(unreached)})

In [ ]:
assert all(score >= THRESHOLD for _, score in admitted)
assert all(score < THRESHOLD for _, score in rejected)
assert len(inspected_ids) + len(unreached) == 26
assert set(inspected_ids).isdisjoint(split.dev_ids)  # compile sees training only
assert set(inspected_ids).isdisjoint(split.holdout_ids)
assert {c for c, _ in admitted} | {c for c, _ in rejected} == set(inspected_ids)

print(f"inspected {len(inspected_ids)}, admitted {len(admitted)}, unreached {len(unreached)}")
print("admission inherits exactly the strengths and blind spots of v1")

## ed-035: a 0.033 lexical loss and a named semantic event

The development-case regression deletes one scope-bearing word. v1 prices it as a tiny overlap change; the v2 rule (replayed with the recorded violated verdict, no judge called) prices the broken condition at 0.700. And the case lives where the filter never looks.

In [ ]:
dev_by_id = {c.case_id: c for c in split.dev}
ed035 = dev_by_id["ed-035"]
good = ed035.reference_rewrite
bad = good.replace("regularly ", "")
assert bad == "Using the filter helps reduce limescale build-up in your kettle."

v1_good = editorial_metric(ed035, good, version="v1").score
v1_bad = editorial_metric(ed035, bad, version="v1").score
v2_replay_bad = min(v1_bad * 0.30, 0.30)  # recorded verdict: regular-use constraint violated

({
    "v1_good": round(v1_good, 4),
    "v1_bad": round(v1_bad, 4),
    "v1_penalty": round(v1_good - v1_bad, 4),
    "v2_replay_bad": round(v2_replay_bad, 4),
    "v2_penalty": round(v1_good - v2_replay_bad, 4),
})

In [ ]:
assert abs((v1_good - v1_bad) - 1 / 30) < 1e-9  # one adverb of twelve reference tokens: 0.40/12
assert abs(v2_replay_bad - 0.29) < 1e-9
assert "ed-035" not in split.train_ids  # dev-side: never enters the bootstrap loop
assert "ed-035" in split.dev_ids

print(f"v1 charges {v1_good - v1_bad:.4f}; v2-replay charges {v1_good - v2_replay_bad:.4f}")
print("changing the compile metric cannot catch what the loop never inspects")

## What we earned

The filter is the mechanism: only traces that satisfy v1 become behavior guidance, and v1's blind spots become the demonstrations' blind spots. The ed-035 regression is development-side evidence, so no compile-metric upgrade evaluated on training traces can touch it — whatever you compile against, something independent must score what came out.

Notebook 12 / Chapter 12 gives the same objective to a stronger search: what happens when the optimizer can rewrite instructions and select against the development set directly?